In [ ]:
!uname -a && cat /etc/*release

In [ ]:
!nvidia-smi

In [ ]:
!nvidia --version

In [ ]:
code5000 = r'''
#define n 5000

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

// Macro for CUDA error checking
#define cudaCheckError() { \
    cudaError_t err = cudaGetLastError(); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(EXIT_FAILURE); \
    } \
}

__global__ void gpuMatMul(double *a, double *b, double *c){
   int row = blockIdx.y * blockDim.y + threadIdx.y;
   int col = blockIdx.x * blockDim.x + threadIdx.x;

   if (row < n && col < n) { // Add bounds check for safety
       double sum = 0.0;
       for (int i = 0; i < n; i++)
          sum += a[row * n + i] * b[i * n + col];
       c[row * n + col] = sum;
   }
}

int main(int argc, char **argv)
{
   cudaEvent_t start, stop, start_dev, stop_dev;
   float elapsed_time_ms_kernel, elapsed_time_ms_total;

   cudaEventCreate(&start); cudaCheckError();
   cudaEventCreate(&stop); cudaCheckError();
   cudaEventCreate(&start_dev); cudaCheckError();
   cudaEventCreate(&stop_dev); cudaCheckError();

   int i, j;

   // allocation on host
   double *a = (double *)malloc(n * n * sizeof(double));
   double *b = (double *)malloc(n * n * sizeof(double));
   double *c = (double *)malloc(n * n * sizeof(double));

   if (a == NULL || b == NULL || c == NULL) {
       fprintf(stderr, "Host memory allocation failed!\n");
       exit(EXIT_FAILURE);
   }

   // initialization
   for (i = 0; i < n; i++)
      for (j = 0; j < n; j++){
         a[i * n + j] = 2.0;
         b[i * n + j] = 3.0;
         c[i * n + j] = 0.0;
      }

   cudaEventRecord(start_dev, 0); cudaCheckError();
   // allocation on device
   double *a_dev, *b_dev, *c_dev;
   cudaMalloc((void **)&a_dev, n * n * sizeof(double)); cudaCheckError();
   cudaMalloc((void **)&b_dev, n * n * sizeof(double)); cudaCheckError();
   cudaMalloc((void **)&c_dev, n * n * sizeof(double)); cudaCheckError();

   // copy data from host to device
   cudaMemcpy(a_dev, a, n * n * sizeof(double), cudaMemcpyHostToDevice); cudaCheckError();
   cudaMemcpy(b_dev, b, n * n * sizeof(double), cudaMemcpyHostToDevice); cudaCheckError();

   dim3 dimGrid(ceil((double)n / 32.0), ceil((double)n / 32.0), 1);
   dim3 dimBlock(32, 32, 1);

   // Print N value for confirmation
   printf("N value: %d\n", n);
   printf("Launching kernel with dimGrid(%d, %d, %d) and dimBlock(%d, %d, %d)\n",
          dimGrid.x, dimGrid.y, dimGrid.z, dimBlock.x, dimBlock.y, dimBlock.z);

   cudaEventRecord(start, 0); cudaCheckError();

   gpuMatMul<<<dimGrid, dimBlock>>>(a_dev, b_dev, c_dev); cudaCheckError();
   cudaDeviceSynchronize(); cudaCheckError(); // Ensure kernel completes before measuring time

   cudaEventRecord(stop, 0); cudaCheckError();
   cudaEventSynchronize(stop); cudaCheckError();

   // data from device to host
   cudaMemcpy(c, c_dev, n * n * sizeof(double), cudaMemcpyDeviceToHost); cudaCheckError();

   cudaEventRecord(stop_dev, 0); cudaCheckError();
   cudaEventSynchronize(stop_dev); cudaCheckError();

   cudaEventElapsedTime(&elapsed_time_ms_kernel, start, stop); cudaCheckError();
   cudaEventElapsedTime(&elapsed_time_ms_total, start_dev, stop_dev); cudaCheckError();

   printf("Result example: c[0][0] = %f\n", c[0]);
   printf("Time to calculate results (kernel only): %f ms.\n", elapsed_time_ms_kernel);
   printf("Total time with I/O (H2D and D2H copies): %f ms.\n", elapsed_time_ms_total);

   cudaFree(a_dev); cudaCheckError();
   cudaFree(b_dev); cudaCheckError();
   cudaFree(c_dev); cudaCheckError();

   free(a);
   free(b);
   free(c);

   return 0;
}
'''

with open('code5000.cu', 'w') as f:
  f.write(code5000)

In [ ]:
!nvcc -arch=sm_75 code5000.cu -o code5000

In [ ]:
!nvprof /content/code5000 > results_5000.txt

In [ ]:
code10000 = r'''
#define n 10000

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

// Macro for CUDA error checking
#define cudaCheckError() { \
    cudaError_t err = cudaGetLastError(); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(EXIT_FAILURE); \
    } \
}

__global__ void gpuMatMul(double *a, double *b, double *c){
   int row = blockIdx.y * blockDim.y + threadIdx.y;
   int col = blockIdx.x * blockDim.x + threadIdx.x;

   if (row < n && col < n) { // Add bounds check for safety
       double sum = 0.0;
       for (int i = 0; i < n; i++)
          sum += a[row * n + i] * b[i * n + col];
       c[row * n + col] = sum;
   }
}

int main(int argc, char **argv)
{
   cudaEvent_t start, stop, start_dev, stop_dev;
   float elapsed_time_ms_kernel, elapsed_time_ms_total;

   cudaEventCreate(&start); cudaCheckError();
   cudaEventCreate(&stop); cudaCheckError();
   cudaEventCreate(&start_dev); cudaCheckError();
   cudaEventCreate(&stop_dev); cudaCheckError();

   int i, j;

   // allocation on host
   double *a = (double *)malloc(n * n * sizeof(double));
   double *b = (double *)malloc(n * n * sizeof(double));
   double *c = (double *)malloc(n * n * sizeof(double));

   if (a == NULL || b == NULL || c == NULL) {
       fprintf(stderr, "Host memory allocation failed!\n");
       exit(EXIT_FAILURE);
   }

   // initialization
   for (i = 0; i < n; i++)
      for (j = 0; j < n; j++){
         a[i * n + j] = 2.0;
         b[i * n + j] = 3.0;
         c[i * n + j] = 0.0;
      }

   cudaEventRecord(start_dev, 0); cudaCheckError();
   // allocation on device
   double *a_dev, *b_dev, *c_dev;
   cudaMalloc((void **)&a_dev, n * n * sizeof(double)); cudaCheckError();
   cudaMalloc((void **)&b_dev, n * n * sizeof(double)); cudaCheckError();
   cudaMalloc((void **)&c_dev, n * n * sizeof(double)); cudaCheckError();

   // copy data from host to device
   cudaMemcpy(a_dev, a, n * n * sizeof(double), cudaMemcpyHostToDevice); cudaCheckError();
   cudaMemcpy(b_dev, b, n * n * sizeof(double), cudaMemcpyHostToDevice); cudaCheckError();

   dim3 dimGrid(ceil((double)n / 32.0), ceil((double)n / 32.0), 1);
   dim3 dimBlock(32, 32, 1);

   // Print N value for confirmation
   printf("N value: %d\n", n);
   printf("Launching kernel with dimGrid(%d, %d, %d) and dimBlock(%d, %d, %d)\n",
          dimGrid.x, dimGrid.y, dimGrid.z, dimBlock.x, dimBlock.y, dimBlock.z);

   cudaEventRecord(start, 0); cudaCheckError();

   gpuMatMul<<<dimGrid, dimBlock>>>(a_dev, b_dev, c_dev); cudaCheckError();
   cudaDeviceSynchronize(); cudaCheckError(); // Ensure kernel completes before measuring time

   cudaEventRecord(stop, 0); cudaCheckError();
   cudaEventSynchronize(stop); cudaCheckError();

   // data from device to host
   cudaMemcpy(c, c_dev, n * n * sizeof(double), cudaMemcpyDeviceToHost); cudaCheckError();

   cudaEventRecord(stop_dev, 0); cudaCheckError();
   cudaEventSynchronize(stop_dev); cudaCheckError();

   cudaEventElapsedTime(&elapsed_time_ms_kernel, start, stop); cudaCheckError();
   cudaEventElapsedTime(&elapsed_time_ms_total, start_dev, stop_dev); cudaCheckError();

   printf("Result example: c[0][0] = %f\n", c[0]);
   printf("Time to calculate results (kernel only): %f ms.\n", elapsed_time_ms_kernel);
   printf("Total time with I/O (H2D and D2H copies): %f ms.\n", elapsed_time_ms_total);

   cudaFree(a_dev); cudaCheckError();
   cudaFree(b_dev); cudaCheckError();
   cudaFree(c_dev); cudaCheckError();

   free(a);
   free(b);
   free(c);

   return 0;
}
'''

with open('code10000.cu', 'w') as f:
  f.write(code10000)

In [ ]:
!nvcc -arch=sm_75 code10000.cu -o code10000

In [ ]:
!nvprof /content/code10000 > results_10000.txt

In [ ]:
code15000 = r'''
#define n 15000

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

// Macro for CUDA error checking
#define cudaCheckError() { \
    cudaError_t err = cudaGetLastError(); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(EXIT_FAILURE); \
    } \
}

__global__ void gpuMatMul(double *a, double *b, double *c){
   int row = blockIdx.y * blockDim.y + threadIdx.y;
   int col = blockIdx.x * blockDim.x + threadIdx.x;

   if (row < n && col < n) { // Add bounds check for safety
       double sum = 0.0;
       for (int i = 0; i < n; i++)
          sum += a[row * n + i] * b[i * n + col];
       c[row * n + col] = sum;
   }
}

int main(int argc, char **argv)
{
   cudaEvent_t start, stop, start_dev, stop_dev;
   float elapsed_time_ms_kernel, elapsed_time_ms_total;

   cudaEventCreate(&start); cudaCheckError();
   cudaEventCreate(&stop); cudaCheckError();
   cudaEventCreate(&start_dev); cudaCheckError();
   cudaEventCreate(&stop_dev); cudaCheckError();

   int i, j;

   // allocation on host
   double *a = (double *)malloc(n * n * sizeof(double));
   double *b = (double *)malloc(n * n * sizeof(double));
   double *c = (double *)malloc(n * n * sizeof(double));

   if (a == NULL || b == NULL || c == NULL) {
       fprintf(stderr, "Host memory allocation failed!\n");
       exit(EXIT_FAILURE);
   }

   // initialization
   for (i = 0; i < n; i++)
      for (j = 0; j < n; j++){
         a[i * n + j] = 2.0;
         b[i * n + j] = 3.0;
         c[i * n + j] = 0.0;
      }

   cudaEventRecord(start_dev, 0); cudaCheckError();
   // allocation on device
   double *a_dev, *b_dev, *c_dev;
   cudaMalloc((void **)&a_dev, n * n * sizeof(double)); cudaCheckError();
   cudaMalloc((void **)&b_dev, n * n * sizeof(double)); cudaCheckError();
   cudaMalloc((void **)&c_dev, n * n * sizeof(double)); cudaCheckError();

   // copy data from host to device
   cudaMemcpy(a_dev, a, n * n * sizeof(double), cudaMemcpyHostToDevice); cudaCheckError();
   cudaMemcpy(b_dev, b, n * n * sizeof(double), cudaMemcpyHostToDevice); cudaCheckError();

   dim3 dimGrid(ceil((double)n / 32.0), ceil((double)n / 32.0), 1);
   dim3 dimBlock(32, 32, 1);

   // Print N value for confirmation
   printf("N value: %d\n", n);
   printf("Launching kernel with dimGrid(%d, %d, %d) and dimBlock(%d, %d, %d)\n",
          dimGrid.x, dimGrid.y, dimGrid.z, dimBlock.x, dimBlock.y, dimBlock.z);

   cudaEventRecord(start, 0); cudaCheckError();

   gpuMatMul<<<dimGrid, dimBlock>>>(a_dev, b_dev, c_dev); cudaCheckError();
   cudaDeviceSynchronize(); cudaCheckError(); // Ensure kernel completes before measuring time

   cudaEventRecord(stop, 0); cudaCheckError();
   cudaEventSynchronize(stop); cudaCheckError();

   // data from device to host
   cudaMemcpy(c, c_dev, n * n * sizeof(double), cudaMemcpyDeviceToHost); cudaCheckError();

   cudaEventRecord(stop_dev, 0); cudaCheckError();
   cudaEventSynchronize(stop_dev); cudaCheckError();

   cudaEventElapsedTime(&elapsed_time_ms_kernel, start, stop); cudaCheckError();
   cudaEventElapsedTime(&elapsed_time_ms_total, start_dev, stop_dev); cudaCheckError();

   printf("Result example: c[0][0] = %f\n", c[0]);
   printf("Time to calculate results (kernel only): %f ms.\n", elapsed_time_ms_kernel);
   printf("Total time with I/O (H2D and D2H copies): %f ms.\n", elapsed_time_ms_total);

   cudaFree(a_dev); cudaCheckError();
   cudaFree(b_dev); cudaCheckError();
   cudaFree(c_dev); cudaCheckError();

   free(a);
   free(b);
   free(c);

   return 0;
}
'''

with open('code15000.cu', 'w') as f:
  f.write(code15000)

In [ ]:
!nvcc -arch=sm_75 code15000.cu -o code15000

In [ ]:
!nvprof /content/code15000 > results_15000.txt

We can look try a tiled version of the code, in order to "speed things up".

In [ ]:
import subprocess
import re

template = r'''
#define n {N}
#define TILE {TILE}

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

// Macro for CUDA error checking
#define cudaCheckError() {{ \
    cudaError_t err = cudaGetLastError(); \
    if (err != cudaSuccess) {{ \
        fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(err)); \
        exit(EXIT_FAILURE); \
    }} \
}}

__global__ void gpuMatMulTiled(double *a, double *b, double *c){{
    __shared__ double As[TILE][TILE];
    __shared__ double Bs[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;

    double sum = 0.0;

    int numTiles = (n + TILE - 1) / TILE;

    for (int t = 0; t < numTiles; t++) {{
        int aCol = t * TILE + threadIdx.x;
        int bRow = t * TILE + threadIdx.y;

        As[threadIdx.y][threadIdx.x] = (row < n && aCol < n) ? a[row * n + aCol] : 0.0;
        Bs[threadIdx.y][threadIdx.x] = (bRow < n && col < n) ? b[bRow * n + col] : 0.0;

        __syncthreads();

        for (int k = 0; k < TILE; k++)
            sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];

        __syncthreads();
    }}

    if (row < n && col < n)
        c[row * n + col] = sum;
}}

int main(int argc, char **argv)
{{
   cudaEvent_t start, stop, start_dev, stop_dev;
   float elapsed_time_ms_kernel, elapsed_time_ms_total;

   cudaEventCreate(&start); cudaCheckError();
   cudaEventCreate(&stop); cudaCheckError();
   cudaEventCreate(&start_dev); cudaCheckError();
   cudaEventCreate(&stop_dev); cudaCheckError();

   int i, j;

   double *a = (double *)malloc(n * n * sizeof(double));
   double *b = (double *)malloc(n * n * sizeof(double));
   double *c = (double *)malloc(n * n * sizeof(double));

   if (a == NULL || b == NULL || c == NULL) {{
       fprintf(stderr, "Host memory allocation failed!\n");
       exit(EXIT_FAILURE);
   }}

   for (i = 0; i < n; i++)
      for (j = 0; j < n; j++){{
         a[i * n + j] = 2.0;
         b[i * n + j] = 3.0;
         c[i * n + j] = 0.0;
      }}

   cudaEventRecord(start_dev, 0); cudaCheckError();
   double *a_dev, *b_dev, *c_dev;
   cudaMalloc((void **)&a_dev, n * n * sizeof(double)); cudaCheckError();
   cudaMalloc((void **)&b_dev, n * n * sizeof(double)); cudaCheckError();
   cudaMalloc((void **)&c_dev, n * n * sizeof(double)); cudaCheckError();

   cudaMemcpy(a_dev, a, n * n * sizeof(double), cudaMemcpyHostToDevice); cudaCheckError();
   cudaMemcpy(b_dev, b, n * n * sizeof(double), cudaMemcpyHostToDevice); cudaCheckError();

   dim3 dimGrid(ceil((double)n / TILE), ceil((double)n / TILE), 1);
   dim3 dimBlock(TILE, TILE, 1);

   printf("N value: %d\n", n);
   printf("TILE value: %d\n", TILE);
   printf("Launching kernel with dimGrid(%d, %d, %d) and dimBlock(%d, %d, %d)\n",
          dimGrid.x, dimGrid.y, dimGrid.z, dimBlock.x, dimBlock.y, dimBlock.z);

   cudaEventRecord(start, 0); cudaCheckError();

   gpuMatMulTiled<<<dimGrid, dimBlock>>>(a_dev, b_dev, c_dev); cudaCheckError();
   cudaDeviceSynchronize(); cudaCheckError();

   cudaEventRecord(stop, 0); cudaCheckError();
   cudaEventSynchronize(stop); cudaCheckError();

   cudaMemcpy(c, c_dev, n * n * sizeof(double), cudaMemcpyDeviceToHost); cudaCheckError();

   cudaEventRecord(stop_dev, 0); cudaCheckError();
   cudaEventSynchronize(stop_dev); cudaCheckError();

   cudaEventElapsedTime(&elapsed_time_ms_kernel, start, stop); cudaCheckError();
   cudaEventElapsedTime(&elapsed_time_ms_total, start_dev, stop_dev); cudaCheckError();

   printf("Result example: c[0][0] = %f\n", c[0]);
   printf("Time to calculate results (kernel only): %f ms.\n", elapsed_time_ms_kernel);
   printf("Total time with I/O (H2D and D2H copies): %f ms.\n", elapsed_time_ms_total);

   cudaFree(a_dev); cudaCheckError();
   cudaFree(b_dev); cudaCheckError();
   cudaFree(c_dev); cudaCheckError();

   free(a);
   free(b);
   free(c);

   return 0;
}}
'''

n_sizes = [5000, 10000, 15000]
tile_sizes = [8, 16, 32]

results = []

for N in n_sizes:
    for TILE in tile_sizes:
        code = template.format(N=N, TILE=TILE)
        src_name = f"code_n{N}_t{TILE}.cu"
        bin_name = f"code_n{N}_t{TILE}"

        with open(src_name, "w") as f:
            f.write(code)

        # Compile
        compile_cmd = ["nvcc", "-arch=sm_75", src_name, "-o", bin_name]
        compile_proc = subprocess.run(compile_cmd, capture_output=True, text=True)

        if compile_proc.returncode != 0:
            print(f"[COMPILE FAILED] n={N}, TILE={TILE}")
            print(compile_proc.stderr)
            continue

        # Run
        run_proc = subprocess.run([f"./{bin_name}"], capture_output=True, text=True)
        output = run_proc.stdout

        # Parse kernel time and total time from output
        kernel_match = re.search(r"kernel only\): ([\d.]+) ms", output)
        total_match = re.search(r"H2D and D2H copies\): ([\d.]+) ms", output)

        kernel_time = float(kernel_match.group(1)) if kernel_match else None
        total_time = float(total_match.group(1)) if total_match else None

        results.append({
            "n": N,
            "TILE": TILE,
            "kernel_ms": kernel_time,
            "total_ms": total_time
        })

# Summary table
with open("results_tiled.txt", "w") as f:
    f.write("=== Summary ===\n")
    f.write("N\tTILE\tkernel_ms\ttotal_ms\n")
    f.write("-" * 40 + "\n")
    for r in results:
        f.write(f"{r['n']}\t{r['TILE']}\t{r['kernel_ms']}\t{r['total_ms']}\n")
        f.write("-" * 40 + "\n")